In [0]:
from pyspark.sql import functions as F

In [0]:
greenhouse_silver = spark.table(
    "job_market.silver.greenhouse_jobs"
)

lever_silver = spark.table(
    "job_market.silver.lever_jobs"
)

print("Greenhouse:", greenhouse_silver.count())
print("Lever:", lever_silver.count())

Greenhouse: 1583
Lever: 638


In [0]:
greenhouse_standardized = greenhouse_silver.select(

    # Lineage
    F.col("source").cast("string").alias("source"),
    F.col("company").cast("string").alias("company"),
    F.col("ingestion_timestamp").cast("timestamp").alias("ingestion_timestamp"),
    F.col("ingestion_date").cast("date").alias("ingestion_date"),

    # Job identity
    F.col("job_id").cast("string").alias("job_id"),
    F.col("job_title").cast("string").alias("job_title"),

    # Classification
    F.col("department").cast("string").alias("department"),
    F.col("office").cast("string").alias("office"),
    F.lit(None).cast("string").alias("team"),
    F.lit(None).cast("string").alias("job_level"),
    F.col("employment_type").cast("string").alias("employment_type"),

    # Location
    F.col("location").cast("string").alias("location"),
    F.col("location_normalized").cast("string").alias("location_normalized"),

    F.when(
        F.col("location").isNotNull(),
        F.array(F.col("location"))
    ).otherwise(
        F.array().cast("array<string>")
    ).alias("all_locations"),

    # Workplace
    F.when(
        F.col("is_remote") == True,
        F.lit("remote")
    ).otherwise(
        F.lit(None).cast("string")
    ).alias("workplace_type"),

    F.col("is_remote").cast("boolean").alias("is_remote"),

    # Description
    F.col("content").cast("string").alias("description"),

    # Dates
    F.col("posted_at").cast("timestamp").alias("posted_at"),
    F.col("updated_at").cast("timestamp").alias("updated_at"),

    # Salary - unavailable from Greenhouse
    F.lit(None).cast("double").alias("salary_min"),
    F.lit(None).cast("double").alias("salary_max"),
    F.lit(None).cast("string").alias("salary_currency"),
    F.lit(None).cast("string").alias("salary_interval"),

    # URLs
    F.col("job_url").cast("string").alias("job_url"),
    F.lit(None).cast("string").alias("apply_url")
)

print("Greenhouse standardization completed")

Greenhouse standardization completed


In [0]:
lever_standardized = lever_silver.select(

    # Lineage
    F.col("source").cast("string").alias("source"),
    F.col("company").cast("string").alias("company"),
    F.col("ingestion_timestamp").cast("timestamp").alias("ingestion_timestamp"),
    F.col("ingestion_date").cast("date").alias("ingestion_date"),

    # Job identity
    F.col("job_id").cast("string").alias("job_id"),
    F.col("job_title").cast("string").alias("job_title"),

    # Classification
    F.col("department").cast("string").alias("department"),
    F.lit(None).cast("string").alias("office"),
    F.col("team").cast("string").alias("team"),
    F.col("job_level").cast("string").alias("job_level"),
    F.col("employment_type").cast("string").alias("employment_type"),

    # Location
    F.col("location").cast("string").alias("location"),
    F.col("location_normalized").cast("string").alias("location_normalized"),
    F.col("all_locations").cast("array<string>").alias("all_locations"),

    # Workplace
    F.col("workplace_type").cast("string").alias("workplace_type"),
    F.col("is_remote").cast("boolean").alias("is_remote"),

    # Description
    F.col("description_plain").cast("string").alias("description"),

    # Dates
    F.col("posted_at").cast("timestamp").alias("posted_at"),
    F.lit(None).cast("timestamp").alias("updated_at"),

    # Salary
    F.col("salary_min").cast("double").alias("salary_min"),
    F.col("salary_max").cast("double").alias("salary_max"),
    F.col("salary_currency").cast("string").alias("salary_currency"),
    F.col("salary_interval").cast("string").alias("salary_interval"),

    # URLs
    F.col("job_url").cast("string").alias("job_url"),
    F.col("apply_url").cast("string").alias("apply_url")
)

print("Lever standardization completed")

Lever standardization completed


In [0]:
print("Greenhouse columns:", len(greenhouse_standardized.columns))
print("Lever columns:", len(lever_standardized.columns))

assert (
    greenhouse_standardized.columns
    == lever_standardized.columns
), "Schemas do not match!"

print("SUCCESS: Both standardized schemas match.")

Greenhouse columns: 25
Lever columns: 25
SUCCESS: Both standardized schemas match.


In [0]:
standardized_jobs = (
    greenhouse_standardized
    .unionByName(lever_standardized)
)

print(
    "Total standardized jobs:",
    standardized_jobs.count()
)

Total standardized jobs: 2221


In [0]:
display(
    standardized_jobs
    .groupBy("source")
    .count()
    .orderBy(F.desc("count"))
)

source,count
greenhouse,1583
lever,638


In [0]:
(
    standardized_jobs
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "job_market.silver.jobs"
    )
)

In [0]:
spark.sql("""
    SELECT
        source,
        COUNT(*) AS job_count
    FROM job_market.silver.jobs
    GROUP BY source
    ORDER BY job_count DESC
""").display()

source,job_count
greenhouse,1583
lever,638
